In [6]:
import pandas as pd 
import numpy as np
from dk_model import DeepKrigingTrainer
import pyarrow.feather as feather
import yaml

In [ ]:
#load model parameters from yaml file
with open('model_pars.yaml', 'r') as file:
    model_params = yaml.safe_load(file)

#Access parameters


mineral = model_params['response']
covariates = model_params['covariates']


#model_params = model_params['model']

learning_rate = model_params['learning_rate']
batch_size = model_params['batch_size']
epochs = model_params['epochs']
cellsize = model_params['cellsize']
layers = model_params['layers']
dropout = model_params['dropout']
activation = model_params['activation']
loss = model_params['loss']
optimizer = model_params['optimizer']

x = model_params['x']
y = model_params['y']
z = model_params['z']

# plotting parameters
cov_labels = model_params['cov_labels']

# file paths
source_dir = model_params['source_dir']
working_dir = model_params['working_dir']
roi_dir = model_params['roi_dir']

In [ ]:
path = f"{working_dir}/combined_data_norm.csv"
deposit_data = pd.read_csv(path, low_memory=False, index_col=False)
deposit_data

,Pd_ppm,Pt_ppm,Ni_ppm,Co_ppm,Cr_ppm,Cu_ppm,Lith1_Code,mid_x_y,mid_y_y,mid_z_y
0,0.000068,0.000585,0.005194,0.002573,0.069948,0.000663,TLAT,0.739576,0.930883,0.998326
1,0.000158,0.001158,0.008235,0.005633,0.144523,0.002306,TLAT,0.739576,0.930883,0.996758
2,0.001186,0.006801,0.015725,0.014674,0.157883,0.004865,TLAT,0.739576,0.930883,0.995190
3,0.003347,0.004966,0.010255,0.059184,0.040190,0.002723,TLAT,0.739576,0.930883,0.993622
4,0.005598,0.005746,0.011867,0.031504,0.029138,0.002925,TLAT,0.739576,0.930883,0.992054
...,...,...,...,...,...,...,...,...,...,...
20727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.881563,0.742479,0.980719
20728,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.853076,0.672712,0.982121
20729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.814283,0.339505,0.965775
20730,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.834120,0.355704,0.975215


In [9]:
covariates + [mineral]

len(deposit_data.dropna(subset=covariates + [mineral]))


#len(deposit_data.dropna(subset=[covariates] + [mineral]))


20665

In [10]:
deposit_data = deposit_data.dropna(subset=covariates + [mineral])

N = len(deposit_data)

In [11]:



lon = deposit_data[x]
lat = deposit_data[y]
az = deposit_data[z]

#num_basis_3_lvl = [10**3, 19**3, 37**3]
#num_basis_2_lvl = [10**3, 19**3]
#num_basis_1_lvl = [10**3]
basis_resolution = [5, 10, 18] # Corresponding to the number of knots in each dimension

num_basis_3_lvl = [basis_resolution[0]**3, basis_resolution[1]**3, basis_resolution[2]**3]
num_basis_2_lvl = [basis_resolution[0]**3, basis_resolution[1]**3]
num_basis_1_lvl = [basis_resolution[0]**3]

num_basis_list = [num_basis_3_lvl, num_basis_2_lvl, num_basis_1_lvl]

phi_arrays = []  

# For each grid
for grid in num_basis_list:
    knots_1dx = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dy = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dz = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    basis_size = 0
    phis = np.zeros((N, sum(grid)))
    
    # For each level of resolution
    for res in range(len(grid)):
        theta = 1 / (grid[res]**(1/3)) * 2.5
        knots_x, knots_y, knots_z = np.meshgrid(knots_1dx[res], knots_1dy[res], knots_1dz[res])
        knots = np.column_stack((knots_x.flatten(), knots_y.flatten(), knots_z.flatten()))
        
        # For each node in the grid
        for i in range(grid[res]):
            d = np.linalg.norm(np.vstack((lon, lat, az)).astype(float).T - knots[i, :], axis=1) / theta
            
            # For each distance of our data to the node i, calculate Wendland kernel
            for j in range(len(d)):
                if 0 <= d[j] <= 1:
                    phis[j, i + basis_size] = (1 - d[j])**6 * (35 * d[j]**2 + 18 * d[j] + 3) / 3
                else:
                    phis[j, i + basis_size] = 0
        
        basis_size += grid[res]
    
    phi_arrays.append(phis)  # Store the phi array for this grid level

# Unpack phi arrays into individual variables
phi_1_lvl, phi_2_lvl, phi_3_lvl = phi_arrays


In [ ]:
phis = [phi_1_lvl, phi_2_lvl, phi_3_lvl]
phi_reduces = {}
dfs = []

phi_columns = deposit_data.columns.tolist()

D = deposit_data.shape[1] # total number of columns in deposit_data 

# Display the list of column names
print(phi_columns[:D])

NameError: name 'phi_1_lvl' is not defined

In [1]:


#deposit_data = deposit_data.dropna(subset=mineral)

for idx, phi in enumerate(phis, start=1):
    idx_zero = np.array([], dtype=int)
    for i in range(phi.shape[1]):
        if np.sum(phi[:, i] != 0) == 0:
            idx_zero = np.append(idx_zero, int(i))

    phi_reduce = np.delete(phi, idx_zero, 1)
    phi_reduces[f"phi_{idx}_lvl_reduce"] = phi_reduce
    
    len_phi_regular = phi.shape[1]
    df_phi_regular = pd.DataFrame(phi, columns=[f'phi_{i}' for i in range(len_phi_regular)])
    dfs.append(df_phi_regular)
    
    len_phi_reduce = phi_reduce.shape[1]
    df_phi_reduce = pd.DataFrame(phi_reduce, columns=[f'phi_{i}' for i in range(len_phi_reduce)])
    dfs.append(df_phi_reduce)
    


NameError: name 'phis' is not defined

In [ ]:
deposit_data = deposit_data.dropna(subset=[mineral] + covariates + phi_columns)

TypeError: can only concatenate str (not "list") to str

In [15]:
deposit_data_list = []
for df in dfs:
    df_reset = df.reset_index(drop=True)
    deposit_data_reset = deposit_data.reset_index(drop=True)

    # Concatenate along columns
    deposit_data_basis = pd.concat([deposit_data_reset, df], axis=1)
    phi_columns = deposit_data_basis.columns[10:].tolist()
    #total_columns = ['CP_Total','PO_Total', 'PY_Total']
    #total_columns = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']
    #covariates = total_columns[:3] + ['RQD_Pct', 'Cr_ppm'] 
    #covariates = total_columns[-2:]
    deposit_data_basis = deposit_data_basis.dropna(subset=[mineral] + covariates + phi_columns)

    deposit_data_list.append(deposit_data_basis)

## Comparison varying the levels of the basis function generating grid

In [ ]:
#first_col = 0 # which column to start from - i.e. which column is the first covariate or contains data?

dfs_names = ['3 levels', '3 levels no 0s', '2 levels', '2 levels no 0s', '1 level', '1 level no 0s']
for df, df_name in zip(deposit_data_list[0:], dfs_names[0:]):
    print(f"\nMetrics for df with {len(df.columns)} columns (grid with {df_name})")
    trainer = DeepKrigingTrainer(df, regular_nn=False, spatial_coords=[x ,y ,z], plot_errors=False, phi_columns=D)
    trainer.train_neural_network()


## Choose the best basis function setup

* '3 levels': deposit_data_list[0]
* '3 levels no 0s':  deposit_data_list[1]
* '2 levels':  deposit_data_list[2]
* '2 levels no 0s':  deposit_data_list[3]
* '1 level':  deposit_data_list[4]
* '1 level no 0s':  deposit_data_list[5]

In [ ]:
#Choose the one with the best R squared

choice = 5

# remember to change the filename to show the levels and whether it has 0s or not
#deposit_data_list[0].to_csv(f'{working_dir}/final_dataset_3levels.csv', index=False)

deposit_data_list[choice].to_csv(f'{working_dir}/final_dataset.csv', index=False)